# 5-4 파라미터 업데이트 코드 흐름

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [2]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [2]:
torch.manual_seed(1)
model = nn.Linear(2, 1)
x = torch.tensor([[1.0, 2.0], [2.0, 1.0]])
y = torch.tensor([[5.0], [4.0]])
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# TODO: 업데이트 전 weight를 복사하세요.
before = model.weight.detach().clone()  # 임시 코드입니다.

pred = model(x)
loss = loss_fn(pred, y)

# TODO: zero_grad -> backward -> step 순서를 작성하세요.
optimizer.zero_grad()
loss.backward()
optimizer.step()

after = model.weight.detach().clone()
print('loss:', float(loss))
print('weight change:', float((after - before).abs().sum()))

loss: 21.484493255615234
weight change: 1.3676142692565918


/tmp/ipykernel_1397/2949416173.py:20: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('loss:', float(loss))


In [3]:
def train_step(model, batch, loss_fn, optimizer):
    x, y = batch
    # TODO: 학습 모드로 전환하세요.
    model.train()
    pred = model(x)
    loss = loss_fn(pred, y)
    # TODO: 업데이트 순서를 완성하세요.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.detach())

model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
batch = (torch.randn(5, 2), torch.randn(5, 1))
print('step loss:', train_step(model, batch, nn.MSELoss(), optimizer))

step loss: 1.061272382736206


In [3]:
torch.manual_seed(7)
model = nn.Linear(1, 1)
x = torch.linspace(-2, 2, 50).view(-1, 1)
y = -3 * x + 0.5
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

def one_step():
    pred = model(x)
    loss = loss_fn(pred, y)
    # TODO: 아래 세 줄의 주석을 풀고 올바른 순서로 배치하세요.

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return float(loss.detach())

loss1 = one_step()
loss2 = one_step()
print('loss1:', loss1, 'loss2:', loss2)

loss1: 14.293404579162598 loss2: 7.603677749633789
